In [33]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME")

APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")

In [34]:
import os
import logging

from opentelemetry._logs import set_logger_provider
from opentelemetry.sdk._logs import (
    LoggerProvider,
    LoggingHandler,
)
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter

logger, logger_provider = None, None

def set_up_logging():
    logger_provider = LoggerProvider()
    set_logger_provider(logger_provider)

    exporter = AzureMonitorLogExporter(connection_string=APPLICATIONINSIGHTS_CONNECTION_STRING)
    logger_provider.add_log_record_processor(BatchLogRecordProcessor(exporter))

    # Attach LoggingHandler to namespaced logger
    handler = LoggingHandler()
    logger = logging.getLogger(__name__)
    logger.addHandler(handler)
    logger.setLevel(logging.NOTSET)
    return logger, logger_provider

def get_logger():
    return logger

def get_logger_provider():
    return logger_provider

# This must be done before any other telemetry calls
logger, logger_provider = set_up_logging()


In [35]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
)
try:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)
    

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
)

def test_coherence( query, answer):
    coherence_evaluator = CoherenceEvaluator(model_config=model_config)
    score = coherence_evaluator(
        query=query, 
        response=answer
    )
    score = 0 if score == None else score["gpt_coherence"]
    return score


def test_groundedness(response, context):
    groundedness_evaluator = GroundednessEvaluator(model_config=model_config)
    score = groundedness_evaluator(
        response=response,
        context=context,
    )
    score = 0 if score == None else score["gpt_groundedness"]
    return score

def test_relevance(query, response, context):
    relevance_eval = RelevanceEvaluator(model_config=model_config)
    score = relevance_eval(
        query=query, 
        response=response,
        context=context,
    )
    score = 0 if score == None else score["gpt_relevance"]
    return score

In [36]:
from azure.search.documents.models import VectorizedQuery
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

credential = AzureKeyCredential(key)
search_client = SearchClient(endpoint=service_endpoint, index_name=index_name, credential=credential)

# Configure OpenAI API
aoai_client = AzureOpenAI(
  azure_endpoint = AZURE_OPENAI_ENDPOINT, 
  api_key=AZURE_OPENAI_KEY,  
  api_version=AZURE_OPENAI_API_VERSION
)

# Function to generate embeddings for title and content fields, also used for query embeddings
def calc_embeddings(text):
    # model = "deployment_name"
    embeddings = aoai_client.embeddings.create(input = [text], model=AZURE_OPENAI_EMBEDDINGS_ADA_DEPLOYMENT_NAME).data[0].embedding
    return embeddings

def get_context_from_vectorBD(query):
    fields = "embedding"
    embedding = calc_embeddings(query)
    vector_query = VectorizedQuery(vector=embedding, k_nearest_neighbors=3, fields=fields)
  
    results = search_client.search(  
        search_text=None,  
        vector_queries= [vector_query],
        select=["content"],
    )  
    answer = ''
    for result in results:   
        answer = answer + result['content']
    return answer

In [37]:
from promptflow.core import Prompty

def run_prompt(prompt_name, query, context) -> str:
    flow = Prompty.load(f"./prompts/{prompt_name}")
    result = flow(question = query, context = context)
    return result

In [45]:
def test_prompt(test_run_id, prompt_name, query_id, query, context):
    answer = run_prompt(prompt_name, query, context)
    entry = {'query_id': query_id, 'query': query, 'prompt_name': prompt_name, 'answer': answer, 'context': context}
    coherence = test_coherence(query, answer)
    groundedness = test_groundedness(answer, context)
    relevance = test_relevance(query, answer, context)
    llm_properties = {
        'llm_version': AZURE_OPENAI_API_VERSION,
        'llm_model': AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
        'temperature': 0,
        'test_run_id': test_run_id,
        'query_id': query_id,
        'prompt_name': prompt_name,
        'coherence': coherence,
        'groundedness': groundedness,
        'relevance': relevance
    }
    logger.warning("Automatic testing", extra=llm_properties)

    return entry

In [47]:
import pandas as pd
import uuid

# evaluating for coherence, groundedness and relevance
# 9 queries * 2 prompts = 18 entries
df = pd.read_csv("./data/queries.csv")
queries_with_answers_df = pd.DataFrame(columns=['query_id', 'query', 'prompt_name', 'answer', 'context'])  

queries_with_answers_list = []
# generate a new guid for each run
test_run_id = str(uuid.uuid4())

for index, row in df.iterrows():

    query_id = row['query_id']
    query = row['query']
    context = get_context_from_vectorBD(query)

    prompt_name = "rag1.prompty"
    entry = test_prompt(test_run_id, prompt_name, query_id, query, context)
    queries_with_answers_list.append(entry)

    prompt_name = "rag2.prompty"
    entry = test_prompt(test_run_id, prompt_name, query_id, query, context)
    queries_with_answers_list.append(entry)  

queries_with_answers_df = pd.DataFrame.from_records(queries_with_answers_list) 
queries_with_answers_df.to_csv("./data/queries_with_answers.csv", index=False)
get_logger_provider().force_flush()

True